# 🔍 Notebook 2 — Testing & Evaluation Pipeline
Scores test images against the trained Memory Banks, computes AUROC/PRO metrics, and generates heatmap visualisations.

**Prerequisites:** Run `01_train.ipynb` first so memory banks exist on Drive.

## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q faiss-cpu tqdm matplotlib scikit-learn scipy

import zipfile, glob, os, logging, shutil, json
DRIVE_DATA = '/content/drive/MyDrive/defects-detection-CV'
MVTEC_ROOT = '/content/data'
os.makedirs(MVTEC_ROOT, exist_ok=True)
for zf in glob.glob(os.path.join(DRIVE_DATA, '*.zip')):
    name = os.path.basename(zf).split('-')[0]
    dest = os.path.join(MVTEC_ROOT, name)
    if os.path.isdir(dest): continue
    print(f'Unzipping {name}...')
    with zipfile.ZipFile(zf, 'r') as z: z.extractall(MVTEC_ROOT)
    if not os.path.isdir(dest):
        for root, dirs, files in os.walk(MVTEC_ROOT):
            if name in dirs and root != MVTEC_ROOT:
                shutil.move(os.path.join(root, name), dest); break
    print(f'  done ✅')

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
logger = logging.getLogger('test')
logger.info('Setup complete.')

## 2 — Configuration

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as models, torchvision.transforms as T
import numpy as np, faiss
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from typing import Dict, List, Tuple, Optional
from tqdm import tqdm
from scipy.ndimage import gaussian_filter, label as connected_components
from sklearn.metrics import roc_auc_score, roc_curve, auc
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

CATEGORIES     = ['bottle', 'carpet', 'screw']
DATA_ROOT      = '/content/data'
OUTPUT_DIR     = '/content/drive/MyDrive/defects-detection/outputs'
MEMORY_BANK_DIR= os.path.join(OUTPUT_DIR, 'memory_banks')
HEATMAP_DIR    = os.path.join(OUTPUT_DIR, 'heatmaps')
IMAGE_SIZE     = 224
BATCH_SIZE     = 32
NUM_WORKERS    = 2
BACKBONE_NAME  = 'wide_resnet50_2'
NN_K           = 1
REWEIGHT_K     = 3
GAUSSIAN_SIGMA = 4.0
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info('Device: %s', DEVICE)

## 3 — Dataset

In [ ]:
class MVTecDataset(Dataset):
    def __init__(self, root_dir, category, split='test', transform=None):
        self.transform = transform or T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)), T.CenterCrop(IMAGE_SIZE), T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        self.mask_transform = T.Compose([T.Resize((IMAGE_SIZE,IMAGE_SIZE)), T.CenterCrop(IMAGE_SIZE), T.ToTensor()])
        self.image_paths, self.mask_paths, self.labels = [], [], []
        split_dir = os.path.join(root_dir, category, split)
        gt_dir = os.path.join(root_dir, category, 'ground_truth')
        for defect_type in sorted(os.listdir(split_dir)):
            defect_dir = os.path.join(split_dir, defect_type)
            if not os.path.isdir(defect_dir): continue
            is_good = defect_type == 'good'
            label = 0 if is_good else 1
            for fname in sorted(os.listdir(defect_dir)):
                if not fname.lower().endswith(('.png','.jpg','.jpeg')): continue
                self.image_paths.append(os.path.join(defect_dir, fname))
                self.labels.append(label)
                if is_good or split == 'train': self.mask_paths.append(None)
                else:
                    mask_name = fname.replace('.png','_mask.png')
                    mp = os.path.join(gt_dir, defect_type, mask_name)
                    self.mask_paths.append(mp if os.path.exists(mp) else None)
        logger.info('[%s/%s] %d images (%d good, %d defective)', category, split, len(self.image_paths), self.labels.count(0), self.labels.count(1))

    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        image = self.transform(Image.open(self.image_paths[idx]).convert('RGB'))
        if self.mask_paths[idx]:
            mask = self.mask_transform(Image.open(self.mask_paths[idx]).convert('L'))
            mask = (mask > 0.5).float()
        else:
            mask = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)
        return image, mask, self.labels[idx], self.image_paths[idx]

## 4 — Feature Extractor

In [ ]:
def extract_layer_features(model, image_batch, hook_dict):
    with torch.no_grad(): _ = model(image_batch)
    return hook_dict['layer2'], hook_dict['layer3']

def locally_aware_patches(feature_map, patch_size=3):
    return F.avg_pool2d(feature_map, kernel_size=patch_size, stride=1, padding=patch_size//2)

def align_and_concat(feat_l2, feat_l3):
    h, w = feat_l2.shape[2], feat_l2.shape[3]
    feat_l3_up = F.interpolate(feat_l3, size=(h,w), mode='bilinear', align_corners=False)
    return torch.cat([feat_l2, feat_l3_up], dim=1)

def flatten_patches(feature_map):
    B, C, H, W = feature_map.shape
    return feature_map.permute(0,2,3,1).reshape(-1, C), (H, W)

## 5 — Scoring

In [ ]:
def compute_patch_distances(test_features, faiss_index, k=1):
    queries = test_features.detach().cpu().numpy().astype(np.float32)
    return faiss_index.search(queries, k)

def reweight_score(distances, neighbor_indices, memory_bank, reweight_k=3):
    mb_np = memory_bank.detach().cpu().numpy().astype(np.float32)
    mb_index = faiss.IndexFlatL2(mb_np.shape[1])
    mb_index.add(mb_np)
    nn_vectors = mb_np[neighbor_indices[:, 0]]
    mb_dists, _ = mb_index.search(nn_vectors, reweight_k)
    weights = np.maximum(mb_dists[:, -1], 1e-8)
    return distances[:, 0] / weights

def patch_scores_to_heatmap(patch_scores, spatial_shape, output_size=224, sigma=4.0):
    H, W = spatial_shape
    hm = torch.tensor(patch_scores.reshape(H,W), dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    hm = F.interpolate(hm, size=(output_size, output_size), mode='bilinear', align_corners=False)
    return gaussian_filter(hm.squeeze().numpy(), sigma=sigma)

def score_image(patch_features, spatial_shape, faiss_index, memory_bank, output_size=224):
    distances, indices = compute_patch_distances(patch_features, faiss_index, k=NN_K)
    patch_scores = reweight_score(distances, indices, memory_bank, REWEIGHT_K)
    heatmap = patch_scores_to_heatmap(patch_scores, spatial_shape, output_size, GAUSSIAN_SIGMA)
    return float(heatmap.max()), heatmap

## 6 — Metrics

In [ ]:
def image_auroc(scores, labels):
    if len(set(labels)) < 2: return 0.0
    return float(roc_auc_score(labels, scores))

def pixel_auroc(heatmaps, gt_masks):
    all_s = np.concatenate([h.flatten() for h in heatmaps])
    all_l = np.concatenate([m.flatten() for m in gt_masks])
    all_l = (all_l > 0.5).astype(np.int32)
    if len(np.unique(all_l)) < 2: return 0.0
    return float(roc_auc_score(all_l, all_s))

def compute_pro(heatmaps, gt_masks, num_thresholds=200, integration_limit=0.3):
    all_scores = np.concatenate([h.flatten() for h in heatmaps])
    thresholds = np.linspace(all_scores.min(), all_scores.max(), num_thresholds)
    pro_fprs, pro_pros = [], []
    for thresh in thresholds:
        overlaps, total_fp, total_tn = [], 0, 0
        for heatmap, mask in zip(heatmaps, gt_masks):
            binary_mask = (mask > 0.5).astype(np.int32)
            prediction = (heatmap >= thresh).astype(np.int32)
            neg_mask = 1 - binary_mask
            total_fp += np.sum(prediction * neg_mask)
            total_tn += np.sum(neg_mask)
            labeled_mask, num_regions = connected_components(binary_mask)
            for rid in range(1, num_regions+1):
                region = (labeled_mask == rid)
                if region.sum() == 0: continue
                overlaps.append(np.sum(prediction[region]) / region.sum())
        pro_fprs.append(total_fp / max(total_tn, 1))
        pro_pros.append(float(np.mean(overlaps)) if overlaps else 0.0)
    sorted_pairs = sorted(zip(pro_fprs, pro_pros))
    fprs = np.array([p[0] for p in sorted_pairs])
    pros = np.array([p[1] for p in sorted_pairs])
    valid = fprs <= integration_limit
    if valid.sum() < 2: return 0.0
    return float(np.trapz(pros[valid], fprs[valid]) / integration_limit)

def evaluate_all(scores, labels, heatmaps, gt_masks):
    return {'image_auroc': image_auroc(scores, labels), 'pixel_auroc': pixel_auroc(heatmaps, gt_masks), 'pro': compute_pro(heatmaps, gt_masks)}

## 7 — Visualisation

In [ ]:
def overlay_heatmap(image, heatmap, alpha=0.4):
    if image.dtype == np.uint8: image = image.astype(np.float32)/255.0
    h_min, h_max = heatmap.min(), heatmap.max()
    hm_norm = (heatmap - h_min)/(h_max - h_min) if h_max - h_min > 1e-8 else np.zeros_like(heatmap)
    colormap = plt.cm.jet(hm_norm)[:,:,:3]
    return np.clip(((1-alpha)*image + alpha*colormap)*255, 0, 255).astype(np.uint8)

def save_qualitative_grid(images, gt_masks, heatmaps, save_path, max_rows=8):
    n = min(len(images), max_rows)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = axes[np.newaxis, :]
    for c, t in enumerate(['Original','Ground Truth','Heatmap','Overlay']): axes[0,c].set_title(t, fontsize=14, fontweight='bold')
    for r in range(n):
        img = np.clip(images[r]*255,0,255).astype(np.uint8) if images[r].dtype != np.uint8 else images[r]
        axes[r,0].imshow(img); axes[r,1].imshow(gt_masks[r], cmap='gray', vmin=0, vmax=1)
        axes[r,2].imshow(heatmaps[r], cmap='jet'); axes[r,3].imshow(overlay_heatmap(images[r], heatmaps[r]))
        for c in range(4): axes[r,c].axis('off')
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)
    logger.info('Grid saved to %s', save_path)

def plot_roc_curve(scores, labels, save_path, title='ROC Curve'):
    if len(set(labels)) < 2: return
    fpr, tpr, _ = roc_curve(labels, scores); roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(7,7))
    ax.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUROC = {roc_auc:.4f}')
    ax.plot([0,1],[0,1], color='gray', lw=1, linestyle='--')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title(title, fontweight='bold')
    ax.legend(loc='lower right'); ax.grid(alpha=0.3)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)

def save_score_histogram(scores, labels, save_path, title='Score Distribution'):
    scores_arr, labels_arr = np.array(scores), np.array(labels)
    fig, ax = plt.subplots(figsize=(8,5))
    good = scores_arr[labels_arr==0]; bad = scores_arr[labels_arr==1]
    if len(good)>0: ax.hist(good, bins=30, alpha=0.6, color='#2ecc71', label=f'Good (n={len(good)})', edgecolor='white')
    if len(bad)>0: ax.hist(bad, bins=30, alpha=0.6, color='#e74c3c', label=f'Defective (n={len(bad)})', edgecolor='white')
    ax.set_xlabel('Anomaly Score'); ax.set_ylabel('Count'); ax.set_title(title, fontweight='bold')
    ax.legend(); ax.grid(alpha=0.3, axis='y')
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches='tight'); plt.close(fig)

## 8 — Run Inference

In [ ]:
def run_patchcore_inference(category):
    logger.info('========== PatchCore Inference: %s ==========', category)
    # 1. Load memory bank
    bank_path = os.path.join(MEMORY_BANK_DIR, f'{category}_memory_bank.pt')
    memory_bank = torch.load(bank_path, map_location=DEVICE, weights_only=True)
    logger.info('Memory bank loaded: %s', memory_bank.shape)

    # 2. Build FAISS index
    vectors = memory_bank.detach().cpu().numpy().astype(np.float32)
    faiss_index = faiss.IndexFlatL2(vectors.shape[1])
    faiss_index.add(vectors)
    logger.info('FAISS index built: %d vectors', faiss_index.ntotal)

    # 3. Load backbone + hooks
    model = getattr(models, BACKBONE_NAME)(weights='IMAGENET1K_V1').to(DEVICE).eval()
    for p in model.parameters(): p.requires_grad = False
    hook_dict = {}
    def _hook(name):
        def fn(m, i, o): hook_dict[name] = o.detach()
        return fn
    h1 = model.layer2.register_forward_hook(_hook('layer2'))
    h2 = model.layer3.register_forward_hook(_hook('layer3'))

    # 4. Load test data
    ds = MVTecDataset(DATA_ROOT, category, 'test')
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # 5. Extract features per image
    inv_norm = T.Normalize(mean=[-0.485/0.229,-0.456/0.224,-0.406/0.225], std=[1/0.229,1/0.224,1/0.225])
    all_features, all_shapes, all_labels, all_masks, all_images = [], [], [], [], []
    for images, masks, labels, paths in tqdm(dl, desc=f'Extracting [{category}]'):
        images = images.to(DEVICE)
        with torch.no_grad():
            fl2, fl3 = extract_layer_features(model, images, hook_dict)
            fl2 = locally_aware_patches(fl2); fl3 = locally_aware_patches(fl3)
            combined = align_and_concat(fl2, fl3)
        for i in range(combined.shape[0]):
            flat, shape = flatten_patches(combined[i].unsqueeze(0))
            all_features.append(flat.cpu()); all_shapes.append(shape)
            all_labels.append(int(labels[i])); all_masks.append(masks[i].cpu())
            img_vis = inv_norm(images[i].cpu()).permute(1,2,0).numpy()
            all_images.append(np.clip(img_vis, 0, 1))

    # 6. Score all images
    image_scores, heatmaps, gt_masks_np = [], [], []
    for feat, shape in tqdm(zip(all_features, all_shapes), total=len(all_features), desc=f'Scoring [{category}]'):
        score, heatmap = score_image(feat, shape, faiss_index, memory_bank, IMAGE_SIZE)
        image_scores.append(score); heatmaps.append(heatmap)
    for m in all_masks:
        gt_masks_np.append(m.squeeze().numpy() if isinstance(m, torch.Tensor) else m)

    # 7. Compute metrics
    metrics = evaluate_all(image_scores, all_labels, heatmaps, gt_masks_np)
    logger.info('[%s] Metrics: %s', category, metrics)

    # 8. Save visualisations
    vis_dir = os.path.join(HEATMAP_DIR, category)
    defective_idx = [i for i, l in enumerate(all_labels) if l == 1]
    grid_idx = defective_idx[:8] if defective_idx else list(range(min(8, len(all_images))))
    save_qualitative_grid([all_images[i] for i in grid_idx], [gt_masks_np[i] for i in grid_idx], [heatmaps[i] for i in grid_idx], os.path.join(vis_dir, f'{category}_qualitative_grid.png'))
    plot_roc_curve(image_scores, all_labels, os.path.join(vis_dir, f'{category}_roc_curve.png'), title=f'ROC — {category}')
    save_score_histogram(image_scores, all_labels, os.path.join(vis_dir, f'{category}_score_histogram.png'), title=f'Scores — {category}')

    # Cleanup hooks
    h1.remove(); h2.remove()
    return metrics

## 9 — Run All Categories

In [ ]:
all_results = {}
for category in CATEGORIES:
    all_results[category] = run_patchcore_inference(category)

# Summary table
print('\n' + '='*60)
print(f'{"Category":<12} | {"Image AUROC":<12} | {"Pixel AUROC":<12} | {"PRO":<8}')
print('-'*60)
for cat, m in all_results.items():
    print(f'{cat:<12} | {m["image_auroc"]:<12.4f} | {m["pixel_auroc"]:<12.4f} | {m["pro"]:<8.4f}')
avg_img = np.mean([m['image_auroc'] for m in all_results.values()])
avg_pix = np.mean([m['pixel_auroc'] for m in all_results.values()])
avg_pro = np.mean([m['pro'] for m in all_results.values()])
print('-'*60)
print(f'{"AVERAGE":<12} | {avg_img:<12.4f} | {avg_pix:<12.4f} | {avg_pro:<8.4f}')
print('='*60)

# Save JSON report
all_results['average'] = {'image_auroc': float(avg_img), 'pixel_auroc': float(avg_pix), 'pro': float(avg_pro)}
report_path = os.path.join(OUTPUT_DIR, 'patchcore_results.json')
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f: json.dump(all_results, f, indent=2)
print(f'\n🎉 Results saved to {report_path}')
print(f'🖼️  Heatmaps saved to {HEATMAP_DIR}/')

## ✅ Done!
Check your Google Drive under `defects-detection/outputs/` for:
- `patchcore_results.json` — all metrics
- `heatmaps/{category}/` — qualitative grids, ROC curves, score histograms